# Merge update performance

Builds:
* `base`: dense columns use the dense kernels, sparse columns the sparse ones
* `sparse`: `ARCTICDB_MERGE_USE_ONLY_SPARSE_CODEPATH=ON`
* `sparse_write_to_segment`: `ARCTICDB_MERGE_USE_ONLY_SPARSE_CODEPATH=ON` and `ARCTICDB_MERGE_USE_WRITE_TO_SEGMENT_TASK=ON`

Each test ran 11 times, each run in its own process. The first run is the warm-up and is dropped. Times are in seconds unless stated otherwise.

## Analysis

### Setup
This notebook has data for the following tests cases.

1. Update and insert with float columns. 10M rows targetm 6M source. 3M updated values 3M inserted values.
2. Update and insert with string columns. 10M rows targetm 6M source. 3M updated values 3M inserted values. String pool size 10k.
3. Update with float columns. 10M rows targetm 6M source. 3M updated values 3M inserted values.
4. Update with string columns. 10M rows targetm 6M source. 3M updated values 3M inserted values. String pool size 10k.
5. Update and insert with float. 10M rows target, 68k source rows. 50% updates, 50% inserts. Each segment of the target has unique index value that spans the whole segment.
6. Update and insert with string. 10M rows target, 68k source rows. 50% updates, 50% inserts. Each segment of the target has unique index value that spans the whole segment.

I planned to have one more test of an index value spanning multiple segments in order to stress the matching but 5 and 6 turned out to be enough. It's pretty slow and it quite memory hungry.

Each test is run 11 times, the first run is discarded as a warmup. Perfetto is used to create custom tracepoints and to analyze the data.

2 Compile time options are introduced in the build. One forcing arcticdb to use the sparse codepath and one forcing the use of write to segment task.

Three builds are compared
1. Baseline (no write to segment task, use dense codepaths for dense data)
2. Use sparse codepaths (do not call write to segment task) used to compare the direct cost of using the sparse codepath
3. Use write to segment task and use the sparse codepath

As a backend I used SeaweedFS allowing parallel writes.

### Is taking the sparse codepath slower (without taking into account write to segment task)

Yes. here's one case where it basically doesn't matter `one_index_value_per_segment_float` (it didn't pass the Welch statistical test, meaning there's not significant difference).

#### Update

Update is a special case. For the dense case we can perform the update inplace. For the sparse case in general we must allocate new column and fill it up. This is why going through the sparse case for updates is particularly slow.

1. **~10% slower for floats** (which is in order of ms for the presented case)
2. **~40% slower for strings** (wich is about ~3 sec)

for the whole merge call

If we zoom in and focus only on the process call (this is done by taking the mean of all process calls in a run and then taking the mean of that over all runs) things are worse.

1. **~14 TIMES slower for floats** this translates into about *70ms* slowdown
2. **~50% slower for strings** this transloates into about *300ms* slodown

Per call to process.

The difference is this large because strings are relatively heavy to work with because of the GIL and the python calls, thus the overhead of creating a new column is not that big. While for floats it's obvious that creating a new column take the majority of time.

The reason the huge difference doesn't show drastically in the end results is that the writes that follow are also slow so pipelining the processing and the write hides the slowdown.

#### Update and insert
##### Randomized data
1. **~5% slower for floats**
2. **~18% slower for strings**

For the two cases with repeated index value, the overhead is dwarfed by the call to match.

Zooming on the process the overhead is higher, but not remotely close to the update. When working with Python strings it's dwarfed by the call to match which dominates the call.

Otherwise it's between 60% and 150% slower. Which translates to diffs > 100ms

### Using WriteToSegmentTask
I the one scenario I didn't test is using `WriteToSegmentTask` and going through the dense codepath. In theory it's the best of both worlds as the floats won't suffer from the sparse codepath down but string processing will be pralell. We can extrapolate the results for this looking at the results for using the sparse-only codepath vs using WriteToSegmentTask + sparse-only codepaths.

`WriteToSegmentTask` makes string processing parallel and improves the runtime dramatically. It takes time proportional to the number of strings in the source (and for example in the update random string case can take half of the function call) but that pays out when there's suffucient parallelization. Even the sparse update path which allocates is faster compared to the dense py string option.

`WriteToSegmentTask` adds negligible overhead to float columns, that can be attributed to noise.

Using `WriteToSegmentTask` + sparse codepath is almost the same as using only the sparse codepath and I'll not discuss it.

#### Speedup with string data
1. Update and insert randomized data: **5 times faster**
2. Update randomized data: **2.7 times faster**
3. Update and insert with repeated index value **252 times faster**

## Conclusion

I sugest taking a hybrid approach.

1. Use `WriteToSegmentTask` to convert all data
2. Keep the two separate codepaths for update and update dense data without performing additional allocations
3. Use the unified (sparse-only) codepath for update and insert

This is a good comprimise between complexity and performance. Handling both sparse and dense data requires lots of compile time polymorhpism which makes the code hard to grasp. It's not worth the 5% speedup we get for using float data.

Allocation of new columns for dense data whose structure we know is a waste of both memory and space and the only thing that saves us is the slow storage.

In [1]:
import pandas as pd
from perfetto.trace_processor import TraceProcessor

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.4f}".format)

SUITES = {
    "base": "benchmark_merge_update_dense",
    "sparse": "benchmark_merge_update_sparse_codepath_for_dense_data",
    "sparse_write_to_segment": "benchmark_merge_update_write_to_segment_task_sparse_codepath",
}
SCENARIOS = ["sparse", "sparse_write_to_segment"]
TESTS = [
    "update_insert_random_float_data",
    "update_insert_random_string_data",
    "update_random_float_data",
    "update_random_string_data",
    "one_index_value_per_segment_float",
    "one_index_value_per_segment_string"
]
UPDATE_TESTS = [t for t in TESTS if t.startswith("update_random")]
UPDATE_INSERT_TESTS = [t for t in TESTS if t not in UPDATE_TESTS]
RUN = ["test", "suite", "run"]

# For the per-column kernels (depth 2) the parent of the parent is the segment's process slice
SLICE_QUERY = """
SELECT s.ts, s.dur, s.name, s.depth, extract_arg(s.arg_set_id, 'debug.column') AS column, p.parent_id AS process_id
FROM slice s LEFT JOIN slice p ON s.parent_id = p.id
"""


def load_slices(path):
    trace = TraceProcessor(trace=path)
    try:
        df = trace.query(SLICE_QUERY).as_pandas_dataframe()
    finally:
        trace.close()
    df = df.infer_objects().astype({"ts": "int64", "dur": "int64", "depth": "int64"}).sort_values("ts")
    runs = df[df["name"] == "merge_internal"].reset_index(drop=True)
    runs["run"] = runs.index
    df = pd.merge_asof(df, runs[["ts", "run"]], on="ts", direction="backward")
    # The chronologically first run is the warm-up
    return df[df["run"] > 0]


slices = pd.concat(
    [load_slices(f"{directory}/test_{test}.zip").assign(suite=suite, test=test) for suite, directory in SUITES.items() for test in TESTS],
    ignore_index=True,
)
slices["dur_s"] = slices["dur"] / 1e9


def per_run_sum(name):
    return slices[slices["name"] == name].groupby(RUN)["dur_s"].sum()


def suite_mean(per_run):
    return per_run.groupby(["test", "suite"]).mean()


def speedup_and_slowdown(means):
    """Splits every (test, scenario) into the cases faster and slower than base."""
    base, new = "mean_baseline_time_s", "new_mean_time_s"
    table = pd.DataFrame(
        [
            {"test": test, "scenario": scenario, base: means[test, "base"], new: means[test, scenario]}
            for test in TESTS
            for scenario in SCENARIOS
        ]
    ).set_index(["test", "scenario"])
    speedup = table[base] / table[new]
    faster = table[speedup > 1].assign(speedup=speedup, **{"speedup_%": 100 * (speedup - 1)})
    slower = table[speedup <= 1].assign(slowdown=1 / speedup, **{"slowdown_%": 100 * (1 / speedup - 1)})
    return faster, slower


def tests_in(table, tests):
    return table[table.index.get_level_values("test").isin(tests)]


total = per_run_sum("merge_internal")

## Total time

Duration of `merge_internal`, mean and standard deviation over the 10 runs.

In [2]:
total_stats = total.groupby(["test", "suite"]).agg(["mean", "std"])
total_table = pd.DataFrame(index=pd.Index(TESTS, name="test"))
for suite in SUITES:
    total_table[f"mean_time_{suite}"] = total_stats["mean"].xs(suite, level="suite")
    total_table[f"time_{suite}_sd"] = total_stats["std"].xs(suite, level="suite")
total_table

,mean_time_base,time_base_sd,mean_time_sparse,time_sparse_sd,mean_time_sparse_write_to_segment,time_sparse_write_to_segment_sd
test,,,,,,
update_insert_random_float_data,2.1944,0.0420,2.3106,0.0231,2.3027,0.0376
update_insert_random_string_data,18.6248,0.2561,22.0711,0.0704,7.9747,0.0504
update_random_float_data,1.5030,0.0303,1.6544,0.0152,1.6498,0.0229
update_random_string_data,6.6109,0.0759,9.5678,0.0879,6.1659,0.0541
one_index_value_per_segment_float,7.8383,0.1090,7.6319,0.1077,7.7074,0.1189
one_index_value_per_segment_string,398.0867,5.9250,395.9095,2.2687,29.6839,0.9466


## Overall Speedup

Cases where the total time is lower than `base`. speedup = base / new, speedup_% = (speedup - 1) × 100.

In [3]:
total_faster, total_slower = speedup_and_slowdown(suite_mean(total))

### Update only

Tests with `not_matched_by_target="do_nothing"`.

In [4]:
tests_in(total_faster, UPDATE_TESTS)

,,mean_baseline_time_s,new_mean_time_s,speedup,speedup_%
test,scenario,,,,
update_random_string_data,sparse_write_to_segment,6.6109,6.1659,1.0722,7.2165


### Update and insert

Tests with `not_matched_by_target="insert"`.

In [5]:
tests_in(total_faster, UPDATE_INSERT_TESTS)

mean_baseline_time_s  new_mean_time_s  speedup  speedup_%
test                               scenario                                                                          
update_insert_random_string_data   sparse_write_to_segment               18.6248           7.9747   2.3355   133.5485
one_index_value_per_segment_float  sparse                                 7.8383           7.6319   1.0271     2.7057
                                   sparse_write_to_segment                7.8383           7.7074   1.0170     1.6995
one_index_value_per_segment_string sparse                               398.0867         395.9095   1.0055     0.5499
                                   sparse_write_to_segment              398.0867          29.6839  13.4108 1,241.0840

## Overall Slowdown

Cases where the total time is higher than `base`. slowdown = new / base, slowdown_% = (slowdown - 1) × 100.

### Update only

Tests with `not_matched_by_target="do_nothing"`.

In [6]:
tests_in(total_slower, UPDATE_TESTS)

mean_baseline_time_s  new_mean_time_s  slowdown  slowdown_%
test                      scenario                                                                            
update_random_float_data  sparse                                 1.5030           1.6544    1.1007     10.0705
                          sparse_write_to_segment                1.5030           1.6498    1.0977      9.7662
update_random_string_data sparse                                 6.6109           9.5678    1.4473     44.7283

### Update and insert

Tests with `not_matched_by_target="insert"`.

In [7]:
tests_in(total_slower, UPDATE_INSERT_TESTS)

mean_baseline_time_s  new_mean_time_s  slowdown  slowdown_%
test                             scenario                                                                            
update_insert_random_float_data  sparse                                 2.1944           2.3106    1.0529      5.2938
                                 sparse_write_to_segment                2.1944           2.3027    1.0494      4.9361
update_insert_random_string_data sparse                                18.6248          22.0711    1.1850     18.5038

## WriteToSegmentTask

Only `sparse_write_to_segment` runs it, once per merge. write_to_segment_% = mean_write_to_segment_time / mean_total_time × 100.

In [8]:
write_to_segment = per_run_sum("WriteToSegmentTask").xs("sparse_write_to_segment", level="suite")
wtst_table = pd.DataFrame(
    {
        "mean_total_time": total.xs("sparse_write_to_segment", level="suite").groupby("test").mean(),
        "mean_write_to_segment_time": write_to_segment.groupby("test").mean(),
        "write_to_segment_time_sd": write_to_segment.groupby("test").std(),
    }
).reindex(TESTS)
wtst_table["write_to_segment_%"] = 100 * wtst_table["mean_write_to_segment_time"] / wtst_table["mean_total_time"]
wtst_table

,mean_total_time,mean_write_to_segment_time,write_to_segment_time_sd,write_to_segment_%
test,,,,
update_insert_random_float_data,2.3027,0.0000,0.0000,0.0016
update_insert_random_string_data,7.9747,4.6919,0.0390,58.8351
update_random_float_data,1.6498,0.0000,0.0000,0.0024
update_random_string_data,6.1659,4.1532,0.0572,67.3579
one_index_value_per_segment_float,7.7074,0.0000,0.0000,0.0005
one_index_value_per_segment_string,29.6839,0.0603,0.0015,0.2033


## Update vs Update sparse

Per-column kernels of the data columns: `update` against `update_sparse` in the update tests, `merge` against `merge_sparse` in the update + insert tests. The `ts` index column uses `merge` in every build and is left out. A segment's time is the sum over its columns. It is averaged over the run's segments, then over the 10 runs.

In [9]:
kernels = slices[(slices["depth"] == 2) & (slices["column"] != "ts")]
per_segment = kernels.groupby(RUN + ["process_id"])["dur_s"].sum()
kernel_means = suite_mean(per_segment.groupby(RUN).mean())
kernel_names = kernels.groupby(["test", "suite"])["name"].first()
kernel_table = pd.DataFrame(index=pd.Index(TESTS, name="test"))
kernel_table["kernels"] = [f"{kernel_names[t, 'base']} vs {kernel_names[t, 'sparse']}" for t in TESTS]
for suite in SUITES:
    kernel_table[f"mean_time_{suite}_s"] = kernel_means.xs(suite, level="suite")
kernel_table

,kernels,mean_time_base_s,mean_time_sparse_s,mean_time_sparse_write_to_segment_s
test,,,,
update_insert_random_float_data,merge vs merge_sparse,0.1104,0.1836,0.1813
update_insert_random_string_data,merge vs merge_sparse,2.2950,2.6947,0.4235
update_random_float_data,update vs update_sparse,0.0055,0.0795,0.0801
update_random_string_data,update vs update_sparse,0.5645,0.8796,0.2092
one_index_value_per_segment_float,merge vs merge_sparse,0.0369,0.0925,0.0924
one_index_value_per_segment_string,merge vs merge_sparse,50.0591,52.9716,0.1980


### Update only

`update` against `update_sparse`, the tests with `not_matched_by_target="do_nothing"`. Same formulas as Overall Speedup and Overall Slowdown.

In [10]:
kernel_faster, kernel_slower = speedup_and_slowdown(kernel_means)

#### Speedup

In [11]:
tests_in(kernel_faster, UPDATE_TESTS)

,,mean_baseline_time_s,new_mean_time_s,speedup,speedup_%
test,scenario,,,,
update_random_string_data,sparse_write_to_segment,0.5645,0.2092,2.6978,169.7771


#### Slowdown

In [12]:
tests_in(kernel_slower, UPDATE_TESTS)

mean_baseline_time_s  new_mean_time_s  slowdown  slowdown_%
test                      scenario                                                                            
update_random_float_data  sparse                                 0.0055           0.0795   14.5038  1,350.3838
                          sparse_write_to_segment                0.0055           0.0801   14.6129  1,361.2873
update_random_string_data sparse                                 0.5645           0.8796    1.5582     55.8192

### Update and insert

`merge` against `merge_sparse`, the tests with `not_matched_by_target="insert"`. Same formulas as Overall Speedup and Overall Slowdown.

#### Speedup

In [13]:
tests_in(kernel_faster, UPDATE_INSERT_TESTS)

,,mean_baseline_time_s,new_mean_time_s,speedup,speedup_%
test,scenario,,,,
update_insert_random_string_data,sparse_write_to_segment,2.2950,0.4235,5.4189,441.8852
one_index_value_per_segment_string,sparse_write_to_segment,50.0591,0.1980,252.8594,"25,185.9441"


#### Slowdown

In [14]:
tests_in(kernel_slower, UPDATE_INSERT_TESTS)

mean_baseline_time_s  new_mean_time_s  slowdown  slowdown_%
test                               scenario                                                                            
update_insert_random_float_data    sparse                                 0.1104           0.1836    1.6633     66.3260
                                   sparse_write_to_segment                0.1104           0.1813    1.6424     64.2374
update_insert_random_string_data   sparse                                 2.2950           2.6947    1.1742     17.4172
one_index_value_per_segment_float  sparse                                 0.0369           0.0925    2.5051    150.5132
                                   sparse_write_to_segment                0.0369           0.0924    2.5022    150.2247
one_index_value_per_segment_string sparse                                50.0591          52.9716    1.0582      5.8181

## Match cost

`match` runs once per segment. match_time is its duration summed over a run's segments, averaged over the 10 runs. match_% = mean match time / mean summed `process` time × 100. Segments run on up to 20 threads in parallel, so match_time can exceed the total time.

In [15]:
match_means = suite_mean(per_run_sum("match"))
process_means = suite_mean(per_run_sum("process"))
match_table = pd.DataFrame(index=pd.Index(TESTS, name="test"))
for suite in SUITES:
    match_table[f"match_time_{suite}"] = match_means.xs(suite, level="suite")
    match_table[f"match_%_{suite}"] = 100 * match_means.xs(suite, level="suite") / process_means.xs(suite, level="suite")
match_table

,match_time_base,match_%_base,match_time_sparse,match_%_sparse,match_time_sparse_write_to_segment,match_%_sparse_write_to_segment
test,,,,,,
update_insert_random_float_data,4.1815,23.1567,4.0447,16.0639,4.2232,16.8612
update_insert_random_string_data,110.7873,32.4428,132.3595,32.8494,5.0532,10.2292
update_random_float_data,3.7936,59.4459,3.5758,26.1361,3.7565,27.4694
update_random_string_data,61.6470,51.9114,84.0830,48.6813,6.1229,21.5910
one_index_value_per_segment_float,133.1266,96.4286,124.0037,92.2939,125.3306,92.3073
one_index_value_per_segment_string,"2,895.0131",36.6380,"2,577.6160",32.7296,530.1620,96.2127
